## 💻 SFT for Code Suggestion
*Experiment: Best run*

---

### 📌 Key Parameters
*   **Rank:** `32`
*   **Alpha:** `32`
*   **Target Gates:** `q`, `k`, `v`, `o`, `gate`, `up`, `down`
*   **Epochs:** `2`
*   **Learning Rate:** `5e-4`

## 1) Import Libraries

In [ ]:
# Check GPU
!nvidia-smi


Mon Sep 21 17:07:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             49W /  400W |   15436MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install unsloth

In [ ]:
import unsloth
import os
from pathlib import Path

import torch
from unsloth import FastLanguageModel
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, DatasetDict
from trl import SFTTrainer, SFTConfig

from huggingface_hub import login, HfApi, upload_folder
import getpass
import datetime

## 2) Config (model + data + hyperparams)


In [ ]:
set_seed(42)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_URL = "magicoder_instruct_dataset.jsonl"
SYSTEM_PROMPT = """
You are an elite AI Backend Engineer. Your sole task is to translate natural language requirements into precise, production-grade Python code for AI systems (e.g., FastAPI, SQLAlchemy, Vector databases, LLM integrations).

Strict Output Rules:
1. Output ONLY the raw Python code.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```python ... ``` or ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the code:" or "Sure!").
4. Enforce production best practices implicitly: utilize async/await for I/O-bound operations, include strict type hints, ensure proper resource management (e.g., context managers for database sessions), and never block the event loop.
"""


# ====== TRAINING ======
MAX_SEQ_LENGTH = 2048
RANK = 32
ALPHA_RANK = 32
GATES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
NUM_EPOCHS = 2
LR = 5e-4
WARMUP_STEPS = 100

# Batch
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8

# Outputs
WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / "artifacts" / "qwen2.5-coder-7B-best-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Memory
LOAD_IN_4BIT = False

# dtype
DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("BASE_MODEL:", BASE_MODEL)
print("DTYPE:", DTYPE, "| LOAD_IN_4BIT:", LOAD_IN_4BIT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


BASE_MODEL: Qwen/Qwen2.5-Coder-7B-Instruct
DTYPE: bfloat16 | LOAD_IN_4BIT: False
GPU: NVIDIA A100-SXM4-40GB


## 3) Load model + Turn on LoRA (Unsloth)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = RANK,
    target_modules = GATES,
    lora_alpha = ALPHA_RANK,
    bias = "none",
    use_gradient_checkpointing = True,
    use_rslora = False,
    loftq_config = None,
    random_state=42
)
model.config.use_cache = False
model.print_trainable_parameters()

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print("Loaded model + LoRA OK")


==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491
Loaded model + LoRA OK


## 4) Load dataset JSONL & Format by chat_template


In [ ]:
raw_dataset = load_dataset("json", data_files=DATASET_URL)

train_test = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": train_test["train"],
    "val": val_test["train"],
    "test": val_test["test"]
})

print(dataset)

sample = dataset["train"][0]
print("\nUser Prompt preview:\n")
print(sample["messages"][0]["content"][:500])
print("\nAssistant Code preview:\n")
print(sample["messages"][1]["content"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1600
    })
    val: Dataset({
        features: ['messages'],
        num_rows: 200
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 200
    })
})

User Prompt preview:

Please create a function that generates text-to-speech (TTS) audio based on a given input text and an optional voice selection. The function should take two parameters: a string `text` representing the content to be converted into speech and a string `voice` which defaults to 'Jessie' if not provided. 

The function should use an external API for TTS conversion and handle the HTTP request using the `requests` library. Ensure that the function checks for successful API response and decodes the au

Assistant Code preview:

def generate_tts(text: str, voice: str='Jessie') -> bytes:
    voice_id = VOICES.get(voice, voice)
    resp = requests.post('https://ttsvibes.com/?/generate', headers={'accept': 'applicatio

In [ ]:
def formatting_prompts_func(examples):
    batch_messages = examples["messages"]
    texts = []

    for messages in batch_messages:
        chat_template_messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]

        chat_template_messages.extend(messages)

        text = tokenizer.apply_chat_template(
            chat_template_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)

Map (num_proc=2):   0%|          | 0/1600 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
print("Dataset size:", len(dataset["train"]))
print("\n===== PREVIEW 2 SAMPLES =====")
for i in range(2):
    print(f"\n--- Sample {i+1} ---\n{dataset['train'][i]['text']}\n...")

Dataset size: 1600

===== PREVIEW 2 SAMPLES =====

--- Sample 1 ---
<|im_start|>system

You are an elite AI Backend Engineer. Your sole task is to translate natural language requirements into precise, production-grade Python code for AI systems (e.g., FastAPI, SQLAlchemy, Vector databases, LLM integrations).

Strict Output Rules:
1. Output ONLY the raw Python code.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```python ... ``` or ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the code:" or "Sure!").
4. Enforce production best practices implicitly: utilize async/await for I/O-bound operations, include strict type hints, ensure proper resource management (e.g., context managers for database sessions), and never block the event loop.
<|im_end|>
<|im_start|>user
Please create a function that generates text-to-speech (TTS) audio based on a given input text and an optional voice selection. The function s

## 5) Train SFT


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["val"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,

    args = SFTConfig(
        gradient_accumulation_steps = GRAD_ACCUM,
        per_device_train_batch_size = PER_DEVICE_BATCH,
        per_device_eval_batch_size = PER_DEVICE_BATCH,
        learning_rate = LR,
        num_train_epochs = NUM_EPOCHS,
        warmup_steps = WARMUP_STEPS,

        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",

        save_strategy = "steps",
        save_steps = 20,
        save_total_limit = 2,

        fp16 = (DTYPE == "float16"),
        bf16 = (DTYPE == "bfloat16"),
        optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = OUTPUT_DIR,

        completion_only_loss = True,
        gradient_checkpointing=True
    ),
)

train_result = trainer.train()
train_result.metrics

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1600 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,600 | Num Epochs = 2 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-pack

Step,Training Loss,Validation Loss
20,1.604922,1.048053
40,0.838962,0.708011
60,0.684910,0.657982
80,0.667021,0.638614
100,0.677129,0.626028
120,0.616693,0.625748
140,0.600835,0.617335
160,0.588643,0.607050
180,0.591483,0.603063
200,0.570180,0.601318


Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-80/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-140/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content

{'train_runtime': 772.9877,
 'train_samples_per_second': 4.14,
 'train_steps_per_second': 0.259,
 'total_flos': 7.921969962620314e+16,
 'train_loss': 0.7440778064727783,
 'epoch': 2.0}

In [ ]:
for callback in trainer.callback_handler.callbacks.copy():
    if "Notebook" in callback.__class__.__name__:
        trainer.remove_callback(callback)

eval_results = trainer.evaluate()
print("===== VAL SET METRICS =====")
eval_results

===== VAL SET METRICS =====


{'eval_loss': 0.6013180017471313,
 'eval_runtime': 10.4631,
 'eval_samples_per_second': 19.115,
 'eval_steps_per_second': 9.557,
 'epoch': 2.0}

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_test_dataset = dataset["test"].map(
    tokenize_function,
    batched=True,
    num_proc=2
)

test_results = trainer.evaluate(
    eval_dataset=tokenized_test_dataset,
    metric_key_prefix="test"
)

print("===== TEST SET METRICS =====")
print(test_results)

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

===== TEST SET METRICS =====
{'test_loss': 0.6202365756034851, 'test_runtime': 10.333, 'test_samples_per_second': 19.355, 'test_steps_per_second': 9.678, 'epoch': 2.0}


## 6) Save LoRA Adapter

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-best-lora/tokenizer_config.json.


Saved LoRA adapter to: /content/artifacts/qwen2.5-coder-7B-best-lora


## 7) Push to Hugging Face

In [ ]:
hf_token = getpass.getpass()
login(token=hf_token)
print("HF login OK")

··········
HF login OK


In [ ]:
api = HfApi()
USERNAME = api.whoami()["name"]

REPO = f"{USERNAME}/Qwen2.5-Coder-7B-Instruct-backend-LoRA"
api.create_repo(REPO, private=False, exist_ok=True)

RepoUrl('https://huggingface.co/hn-minh/Qwen2.5-Coder-7B-Instruct-backend-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='hn-minh/Qwen2.5-Coder-7B-Instruct-backend-LoRA')

In [ ]:
ckpt_path = "./artifacts/qwen2.5-coder-7B-best-lora"
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = AutoModelForCausalLM.from_pretrained(ckpt_path)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

In [ ]:
model.push_to_hub(
    REPO,
    commit_message="Add model ckpt",
)

tokenizer.push_to_hub(
    REPO,
    commit_message="Add tokenizer ckpt",
)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  616kB /  162MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp_5qnf627/tokenizer.json:   0%|          | 27.8kB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/hn-minh/Qwen2.5-Coder-7B-Instruct-backend-LoRA/commit/3f90f693c5cf7cf7a1b8ba28fee936e7c3ac97af', commit_message='Add tokenizer ckpt', commit_description='', oid='3f90f693c5cf7cf7a1b8ba28fee936e7c3ac97af', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hn-minh/Qwen2.5-Coder-7B-Instruct-backend-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='hn-minh/Qwen2.5-Coder-7B-Instruct-backend-LoRA'), pr_revision=None, pr_num=None)

## 8) Sanity check


In [ ]:
model.to(DEVICE)
FastLanguageModel.for_inference(model)

user_question = "Write a FastAPI endpoint (async def) that accepts a long piece of text, uses Hugging Face AutoTokenizer to count the number of tokens, and returns the result. The system needs to handle thousands of concurrent requests."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(DEVICE)

outputs = model.generate(
    **inputs,
    max_new_tokens=3000,
    use_cache=True,
    temperature=0.1,
)

prompt_length = inputs["input_ids"].shape[1]
response_tokens = outputs[0][prompt_length:]
response = tokenizer.decode(response_tokens, skip_special_tokens=True)

print("===== TEST INFERENCE =====")
print(f"User: {user_question}")
print(f"Code: {response.strip()}")